<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">7 - Florestas Públicas</span>
<hr style="border:1px solid #0077b9;">

As Florestas Públicas designam áreas com cobertura florestal pertencentes ao governo brasileiro (federal, estadual, municipal ou distrital). Essas áreas podem estar localizadas em todos os biomas brasileiros: Amazônia, Cerrado, Mata Atlântica, Pantanal, Caatinga e Pampa. As florestas públicas podem ser naturais, plantadas ou a combinação das duas. A administração das florestas públicas assume a premissa de que elas pertencem ao domínio público e, portanto, devem ser preservadas e exploradas de maneira sustentável para o benefício da sociedade. Nesse contexto, a Lei nº 11.284, de 2 de março de 2006

De acordo com a Resolução CMN Nº 5.193/2024:

> 14 - Não será concedido crédito rural a empreendimento situado em imóvel rural total ou parcialmente inserido em
Floresta Pública Tipo B (Não Destinada) registrada no Cadastro Nacional de Florestas Públicas (CNFP) do
Serviço Florestal Brasileiro (SFB). (Res CMN 5.193 art 1º)

> 15 - A vedação de que trata o item 14 não abrange: (Res CMN 5.268 art 1º) (*)
a) os imóveis rurais matriculados em registro de imóveis; e
b) os imóveis com até quinze módulos fiscais, desde que seja mantida a vegetação nativa na área de Floresta
Pública Tipo B e a área ocupada pelo empreendimento a ser financiado não esteja inserida, total ou
parcialmente, na respectiva Floresta Pública.

Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 7:

![car](https://i.imgur.com/fNfaMXO.png "Florestas Públicas")

Prática: vamos simular a verificação de sobreposição.




**Configuração e Exemplo que NÂO atende a norma**

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

In [ ]:
empreendimento = {
    "geometria": Polygon([(0, 0), (0, 2), (2, 2), (2, 0)]),
    "possuí_matricula": True,
    "modulos_fiscais": 20,
    "floresta_mantida": True,
    "producao_fora_da_floresta": True
}

In [ ]:
lista_florestas_publicas = [
    {"nome": "Floresta Nacional A", "geometria": Polygon([(1, 1), (1, 3), (3, 3), (3, 1)])},
    {"nome": "Reserva Extrativista B", "geometria": Polygon([(10, 10), (10, 12), (12, 12), (12, 10)])}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, floresta in enumerate(lista_florestas_publicas):
    x_flor, y_flor = floresta["geometria"].exterior.xy

    label = 'Floresta Pública' if i == 0 else None

    ax.plot(x_flor, y_flor, color='#2ca02c', linewidth=2, label=label)
    ax.fill(x_flor, y_flor, color='#2ca02c', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Florestas Públicas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, floresta in enumerate(lista_florestas_publicas):

    # Verifica se a geometria do empreendimento possui sobreposição com a floresta pública.
    if empreendimento["geometria"].intersects(floresta["geometria"]):

        # Verifica se o imóvel possui matrícula devidamente registrada em cartório.
        if empreendimento["possuí_matricula"]:

            # Verifica se o imóvel é classificado como pequeno (até 15 módulos fiscais).
            if empreendimento["modulos_fiscais"] <= 15:

                # Verifica se a floresta pública é mantida e a atividade ocorre fora dela.
                if empreendimento["floresta_mantida"] and empreendimento["producao_fora_da_floresta"]:
                    # Atende aos critérios excepcionais para pequenos imóveis matriculados.
                    pass
                else:
                    imovel_atende_norma = False
                    motivo_impedimento = "atividade econômica ocorre dentro da floresta ou a mesma não é mantida."
                    break

            # Caso o imóvel possua matrícula, mas não seja pequeno (mais de 15 módulos).
            else:
                imovel_atende_norma = False
                motivo_impedimento = "imóvel não é pequeno (superior a 15 módulos fiscais)."
                break

        # Caso o imóvel rural não possua matrícula no cartório.
        else:
            imovel_atende_norma = False
            motivo_impedimento = "imóvel não possui matrícula no cartório de registro de imóveis."
            break

    # Caso não exista nenhuma sobreposição com a floresta pública atual.
    else:
        pass

# Verifica se o imóvel passou por todas as etapas de validação de florestas públicas.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
else:
    print(f"O empreendimento não atende a norma porque: {motivo_impedimento}")

**Configuração e Exemplo atende a norma**

In [ ]:
empreendimento = {
    "geometria": Polygon([(4, 4), (4, 8), (8, 8), (8, 4)]),
    "possuí_matricula": True,
    "modulos_fiscais": 8,
    "floresta_mantida": True,
    "producao_fora_da_floresta": True
}

In [ ]:
lista_florestas_publicas = [
    {"nome": "Floresta Nacional A", "geometria": Polygon([(1, 1), (1, 3), (3, 3), (3, 1)])},
    {"nome": "Reserva Extrativista B", "geometria": Polygon([(10, 10), (10, 12), (12, 12), (12, 10)])}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, floresta in enumerate(lista_florestas_publicas):
    x_flor, y_flor = floresta["geometria"].exterior.xy

    label = 'Floresta Pública' if i == 0 else None

    ax.plot(x_flor, y_flor, color='#2ca02c', linewidth=2, label=label)
    ax.fill(x_flor, y_flor, color='#2ca02c', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Florestas Públicas')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, floresta in enumerate(lista_florestas_publicas):

    # Verifica se a geometria do empreendimento possui sobreposição com a floresta pública.
    if empreendimento["geometria"].intersects(floresta["geometria"]):

        # Verifica se o imóvel possui matrícula devidamente registrada em cartório.
        if empreendimento["possuí_matricula"]:

            # Verifica se o imóvel é classificado como pequeno (até 15 módulos fiscais).
            if empreendimento["modulos_fiscais"] <= 15:

                # Verifica se a floresta pública é mantida e a atividade ocorre fora dela.
                if empreendimento["floresta_mantida"] and empreendimento["producao_fora_da_floresta"]:
                    # Atende aos critérios excepcionais para pequenos imóveis matriculados.
                    pass
                else:
                    imovel_atende_norma = False
                    motivo_impedimento = "atividade econômica ocorre dentro da floresta ou a mesma não é mantida."
                    break

            # Caso o imóvel possua matrícula, mas não seja pequeno (mais de 15 módulos).
            else:
                imovel_atende_norma = False
                motivo_impedimento = "imóvel não é pequeno (superior a 15 módulos fiscais)."
                break

        # Caso o imóvel rural não possua matrícula no cartório.
        else:
            imovel_atende_norma = False
            motivo_impedimento = "imóvel não possui matrícula no cartório de registro de imóveis."
            break

    # Caso não exista nenhuma sobreposição com a floresta pública atual.
    else:
        pass

# Verifica se o imóvel passou por todas as etapas de validação de florestas públicas.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
else:
    print(f"O empreendimento não atende a norma porque: {motivo_impedimento}")